# Classical

In [ ]:
import numpy as np
# np.random.seed(67)

In [ ]:
# 1. Generate example data
mu = 0.0
sigma = 1.0

n_wavelets = 2**6
n_parameters = 5

wavelets_t1 = np.random.normal(mu, sigma, size=(n_parameters, n_wavelets))

# Make t2 somewhat correlated with t1, so the test has a nontrivial cross-covariance matrix
wavelets_t2 = 0.6 * wavelets_t1 + 0.7 * np.random.normal(mu, sigma, size=(n_parameters, n_wavelets))

In [ ]:
# 2. Compute standard cross-covariance across wavelets
nm1 = n_wavelets - 1

X = wavelets_t1 - wavelets_t1.mean(axis=1, keepdims=True)
Y = wavelets_t2 - wavelets_t2.mean(axis=1, keepdims=True)

C = (X @ Y.T) / nm1

print("Cross-covariance shape:", C.shape)
print("Cross-covariance:", C)

# Cross-covariance itself need not be PSD
Sigma = C @ C.T

# Optional: normalize the typical noise variance
# Set step_scale to control the overall step size
step_scale = 0.1

mean_variance = np.trace(Sigma) / n_parameters

if mean_variance > 1E-14:
    Sigma = Sigma / mean_variance

Sigma *= step_scale**2
print("Target covariance:\n", Sigma)

# Symmetric eigendecomposition handles PSD matrices, including singular matrices
eigvals, eigvecs = np.linalg.eigh(Sigma)
eigvals = np.clip(eigvals, 0.0, None)

L = eigvecs @ np.diag(np.sqrt(eigvals))

In [ ]:
# 3. Generate zero-mean noise by a CLT construction
n_rounds = 256

# Independent Rademacher variables: equally likely -1,+1
signs = 2 * np.random.randint(0, 2, size=(n_rounds, n_parameters, n_wavelets)) - 1

# Sum and scale: each wavelet's noise vector has theoretical covariance approximately/exactly Sigma
z = signs.sum(axis=0) / np.sqrt(n_rounds)

# Correlate the parameter perturbations.
epsilon = L @ z

print("Perturbation shape:", epsilon.shape)
print("Empirical perturbation mean:", epsilon.mean())

In [ ]:
# 4. Apply the random walk update
wavelets_t3 = wavelets_t2 + epsilon

# Empirical covariance of the generated noise, estimated across wavelets
epsilon_centered = epsilon - epsilon.mean(axis=1, keepdims=True)
empirical_cov = epsilon_centered @ epsilon_centered.T/ (n_wavelets - 1)

print("Empirical noise covariance:\n", empirical_cov)
print("Updated wavelets shape:", wavelets_t3.shape)

In [ ]:
eps = 1E-12

A = empirical_cov
B = Sigma

# A. Normalized Hilbert-Schmidt similarity (1 is ideal)
denom = np.linalg.norm(A, "fro") * np.linalg.norm(B, "fro")
hs_similarity = (
    np.vdot(A, B).real / denom if denom > eps else np.nan
)

# B. Relative Frobenius error (0 is ideal)
relative_error = (
    np.linalg.norm(A - B, "fro") /
    max(np.linalg.norm(B, "fro"), eps)
)

# C. Absolute Frobenius distance (0 is ideal)
frobenius_distance = np.linalg.norm(A - B, "fro")

print("HS similarity:", hs_similarity)
print("Relative Frobenius error:", relative_error)
print("Frobenius distance:", frobenius_distance)

# Quantum

In [5]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
import numpy as np


def state_controlled_walk(n_bits, theta=0.2):
    """
    Coherent state-dependent walk primitive.

    Registers:
      current  : current parameter bitstring
      previous : previous parameter bitstring
      diff     : XOR of corresponding bits
      coin     : update coin

    No measurements or classical covariance calculations
    occur in the circuit.
    """
    current = QuantumRegister(n_bits, "current")
    previous = QuantumRegister(n_bits, "previous")
    diff = QuantumRegister(n_bits, "diff")
    coin = QuantumRegister(1, "coin")

    qc = QuantumCircuit(current, previous, diff, coin)

    # Illustrative input preparation only.
    # Replace this with your actual joint state preparation.
    for i in range(n_bits):
        qc.h(current[i])
        qc.h(previous[i])

    # Compute bitwise disagreement reversibly.
    for i in range(n_bits):
        qc.cx(current[i], diff[i])
        qc.cx(previous[i], diff[i])

    # Each disagreement bit controls the update coin.
    # This makes transition behavior depend on the input state.
    for i in range(n_bits):
        qc.cry(theta, diff[i], coin[0])

    # Uncompute disagreement, retaining the coin's
    # correlation with the data registers.
    for i in reversed(range(n_bits)):
        qc.cx(previous[i], diff[i])
        qc.cx(current[i], diff[i])

    return qc

def build_walk(n_bits, theta):
    current = QuantumRegister(n_bits, "current")
    previous = QuantumRegister(n_bits, "previous")
    diff = QuantumRegister(n_bits, "diff")
    coin = QuantumRegister(1, "coin")

    qc = QuantumCircuit(current, previous, diff, coin)

    # Uniform input superposition (test input).
    for i in range(n_bits):
        qc.h(current[i])
        qc.h(previous[i])

    # XOR current and previous.
    for i in range(n_bits):
        qc.cx(current[i], diff[i])
        qc.cx(previous[i], diff[i])

    # Apply state-dependent coin rotations.
    for i in range(n_bits):
        qc.cry(theta, diff[i], coin[0])

    # Uncompute difference ancillas.
    for i in reversed(range(n_bits)):
        qc.cx(previous[i], diff[i])
        qc.cx(current[i], diff[i])

    return qc


def benchmark(n_bits=3, theta=0.2):
    qc = build_walk(n_bits, theta)
    sv = Statevector.from_instruction(qc)

    # Qiskit qubit order:
    # current: 0..n-1
    # previous: n..2n-1
    # diff: 2n..3n-1
    # coin: 3n
    probs = sv.probabilities()

    # Exact coin activation probability from the statevector.
    p_coin_qiskit = 0.0
    for idx, p in enumerate(probs):
        coin_bit = (idx >> (3 * n_bits)) & 1
        if coin_bit:
            p_coin_qiskit += p

    # Classical reference:
    # For Hamming distance h, the coin rotation angle is h*theta,
    # so P(coin=1 | h) = sin^2(h*theta/2).
    p_coin_classical = 0.0
    total_inputs = 2 ** (2 * n_bits)

    for x in range(2**n_bits):
        for y in range(2**n_bits):
            h = (x ^ y).bit_count()
            p_coin_classical += (
                np.sin(h * theta / 2)**2 / total_inputs
            )

    print("Qiskit coin probability:", p_coin_qiskit)
    print("Classical reference:", p_coin_classical)
    print("Absolute error:",
          abs(p_coin_qiskit - p_coin_classical))
    print("Statevector norm:", np.linalg.norm(sv.data))

    return p_coin_qiskit, p_coin_classical

benchmark(n_bits=3, theta=0.2)

Qiskit coin probability: 0.029455104347371322
Classical reference: 0.029455104347371332
Absolute error: 1.0408340855860843e-17
Statevector norm: 0.9999999999999994


(np.float64(0.029455104347371322), np.float64(0.029455104347371332))